# Predict Residential EV Charging Loads

In [1]:
import numpy as np
import pandas as pd

### Load, Inspect and Merge Datasets

The electric vehicle (EV) charging data come from various residential apartment buildings in Norway. The data includes specific user and garage information, plug-in and plug-out times, charging loads and the dates of the charging sessions.

In [4]:
ev_charging_reports = pd.read_csv('datasets/EV charging reports.csv')
ev_charging_reports.head()

,session_ID,Garage_ID,User_ID,User_private,Shared_ID,Start_plugin,Start_plugin_hour,End_plugout,End_plugout_hour,El_kWh,...,month_plugin_Nov,month_plugin_Oct,month_plugin_Sep,weekdays_plugin_Friday,weekdays_plugin_Monday,weekdays_plugin_Saturday,weekdays_plugin_Sunday,weekdays_plugin_Thursday,weekdays_plugin_Tuesday,weekdays_plugin_Wednesday
0,1,AdO3,AdO3-4,1.0,NaN,21.12.2018 10:20,21.12.2018 10:00,21.12.2018 10:23,10.0,"0,3",...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,AdO3,AdO3-4,1.0,NaN,21.12.2018 10:24,21.12.2018 10:00,21.12.2018 10:32,10.0,"0,87",...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,AdO3,AdO3-4,1.0,NaN,21.12.2018 11:33,21.12.2018 11:00,21.12.2018 19:46,19.0,"29,87",...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,AdO3,AdO3-2,1.0,NaN,22.12.2018 16:15,22.12.2018 16:00,23.12.2018 16:40,16.0,"15,56",...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,5,AdO3,AdO3-2,1.0,NaN,24.12.2018 22:03,24.12.2018 22:00,24.12.2018 23:02,23.0,"3,62",...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


<details><summary style="display:list-item; font-size:16px; color:blue;">structure of the dataset</summary>

- **session_ID** - the unique id for each EV charging session
- **Garage_ID** - the unique id for the garage of the apartment
- **User_ID** - the unique id for each user
- **User_private** - 1.0 indicates private charge point spaces and 0.0 indicates shared charge point spaces
- **Shared_ID** - the unique id if shared charge point spaces are used
- **Start_plugin** - the plug-in date and time in the format (day.month.year hour:minute)
- **Start_plugin_hour** - the plug-in date and time rounded to the start of the hour
- **End_plugout** - the plug-out date and time in the format (day.month.year hour:minute)
- **End_plugout_hour** - the start of the hour of the `End_plugout` hour
- **El_kWh** - the charged energy in kWh (charging loads)
- **Duration_hours** - the duration of the EV connection time per session
- **Plugin_category** - the plug-in time categorized by early/late night, morning, afternoon, and evening
- **Duration_category** - the plug-in duration categorized by 3 hour groups
- **month_plugin_{month}** - the month of the plug-in session
- **weekdays_plugin_{day}** - the day of the week of the plug-in session

`traffic_reports` dataset contains the hourly local traffic density counts at 5 nearby traffic locations.

In [5]:
traffic_reports = pd.read_csv('datasets/Local traffic distribution.csv')
traffic_reports.head()

,Date_from,Date_to,Kroppan_bru_traffic,Moholtlia_traffic,Selsbakk_traffic,Moholt_rampe_2_traffic,Jonsvannsveien_vest_steinanvegen_traffic
0,01.12.2018 00:00,01.12.2018 01:00,639,0,0,4,144
1,01.12.2018 01:00,01.12.2018 02:00,487,153,115,21,83
2,01.12.2018 02:00,01.12.2018 03:00,408,85,75,10,69
3,01.12.2018 03:00,01.12.2018 04:00,282,89,56,8,39
4,01.12.2018 04:00,01.12.2018 05:00,165,64,34,3,25


<details><summary style="display:list-item; font-size:16px; color:blue;">structure of the dataset</summary>

- **Date_from** - the starting time in the format (day.month.year hour:minute)
- **Date_to** - the ending time in the format (day.month.year hour:minute)
- **Location 1 to 5** - contains the number of vehicles each hour at a specified traffic location.


The same charging location may charge at different rates depending on the number of cars being charged, so this traffic data might help the model out.

Merge the `ev_charging_reports` and `traffic_reports` datasets together:

In [6]:
ev_charging_traffic = pd.merge(ev_charging_reports, traffic_reports,
                    left_on='Start_plugin_hour', right_on='Date_from')

In [7]:
ev_charging_traffic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6833 entries, 0 to 6832
Data columns (total 39 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   session_ID                                6833 non-null   int64  
 1   Garage_ID                                 6833 non-null   object 
 2   User_ID                                   6833 non-null   object 
 3   User_private                              6833 non-null   float64
 4   Shared_ID                                 1399 non-null   object 
 5   Start_plugin                              6833 non-null   object 
 6   Start_plugin_hour                         6833 non-null   object 
 7   End_plugout                               6833 non-null   object 
 8   End_plugout_hour                          6833 non-null   float64
 9   El_kWh                                    6833 non-null   object 
 10  Duration_hours                      

We see that there are 39 columns and 6,833 rows in our merged dataset.

Some notable things we might have to address:

- We expected columns like `El_kWh` and `Duration_hours` to be floats but they are actually object data types.

- There are many identifying columns like `session_ID` and `User_ID` that might not be useful for training.

## Data Cleaning and Preparation

Dropping columns that won't be used for training. These include
- ID columns
- columns with lots of missing data
- non-numeric columns (for now, since we haven't yet covered using non-numeric data in neural networks)

In [8]:
ev_charging_traffic = ev_charging_traffic.drop(['session_ID', 'Garage_ID', 'User_ID', 
                                                'Shared_ID', 'Plugin_category', 'Duration_category', 
                                                'Start_plugin', 'Start_plugin_hour', 'End_plugout', 
                                                'End_plugout_hour', 'Date_from', 'Date_to'], axis=1)

Earlier we saw that the `El_kWh` and `Duration_hours` columns were object data types. Upon further inspection, we see that the reason is that the data is following European notation where commas `,` are used as decimals instead of periods.

Replace `,` with `.` in these three columns:

In [9]:
ev_charging_traffic['El_kWh'] = ev_charging_traffic['El_kWh'].str.replace(',', '.')
ev_charging_traffic['Duration_hours'] = ev_charging_traffic['Duration_hours'].str.replace(',', '.')

Next, convert the data types of all the columns of `ev_charging_traffic` to floats.

In [10]:
ev_charging_traffic = ev_charging_traffic.astype('float')

## Train Test Split

In [11]:
X = ev_charging_traffic.drop('El_kWh', axis=1)
y = ev_charging_traffic.El_kWh

In [12]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=2)

## Linear Regression Baseline

compare our neural network to a basic linear regression. After all, if a basic linear regression works just as well, there's no need for the neural network.

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

In [15]:
y_pred = lr.predict(X_test)

from sklearn.metrics import mean_squared_error
test_mse = mean_squared_error(y_test, y_pred)
test_mse

131.4188163356643

Mean squared error is around `131.4`. If we take the square root, we have about `11.5`. One way of interpreting this is to say that the linear regression, on average, is off by `11.5 kWh`.

## Train a Neural Network Using PyTorch

In [16]:
import torch
from torch import nn
from torch import optim

In [17]:
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float).view(-1, 1)

X_test_tensor = torch.tensor(X_test.values, dtype=torch.float)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float).view(-1, 1)

In [19]:
X_train.shape

(5466, 26)

In [20]:
# set a random seed
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(26, 56),
    nn.ReLU(),
    nn.Linear(56, 26),
    nn.ReLU(),
    nn.Linear(26, 1))

In [21]:
loss = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0007)

In [22]:
num_epochs = 3000
for epoch in range(num_epochs):
    predictions = model(X_train_tensor) 
    MSE = loss(predictions, y_train_tensor) 
    MSE.backward()
    optimizer.step() 
    optimizer.zero_grad()
    
    # keep track of the loss every 500 epochs during training
    if (epoch + 1) % 500 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], MSE Loss: {MSE.item()}')

Epoch [500/3000], MSE Loss: 146.75697326660156
Epoch [1000/3000], MSE Loss: 139.6661834716797
Epoch [1500/3000], MSE Loss: 118.94923400878906
Epoch [2000/3000], MSE Loss: 116.21517181396484
Epoch [2500/3000], MSE Loss: 110.67818450927734
Epoch [3000/3000], MSE Loss: 107.02542877197266


In [23]:
torch.save(model, 'models/model.pth')

In [24]:
model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor)
    test_loss = loss(predictions, y_test_tensor)

print('Neural Network - Test Set MSE:', test_loss.item())

Neural Network - Test Set MSE: 122.73201751708984


The same model was trained for 4500 epochs locally and saved as `models/model4500.pth`.

How well does the longer-trained model perform?

In [25]:
loaded_model = torch.load('models/model4500.pth')
with torch.no_grad():
    predictions = loaded_model(X_test_tensor)
    test_loss = loss(predictions, y_test_tensor)

print('Neural Network - Test Set MSE:', test_loss.item())

Neural Network - Test Set MSE: 115.21600341796875


The increased training improved our test loss to about `115.2`, a full `12%` improvement on our linear regression baseline. So the nonlinearity introduced by the neural network actually helped us out.

Some things to investigate further include:
- explore different ways to clean and prepare the data
- we added traffic data, but there's no guarantee that more data converts to a better model. Test out different sets of input columns.
- test out different number of nodes in the hidden layers, activation functions, and learning rates
- train on a larger number of epochs 